<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 附录 A1 · 扩展 Source、Artifact 与 Trigger

前 22 篇使用产品已经组装好的接口。这篇面向要开发新组件的人：把一次真实检查结果定义为新 Source，用自定义 Artifact 保存检查报告，用纯 Trigger 决定何时产生报告。

无需模型。我们使用公开的核心类型和内置关系存储扩展点；这不会自动给 Server 增加新的 HTTP 路由、召回策略或审核语义。那些服务能力需要另行组装。本篇演示类型、来源、纯策略和真实持久化，不另造一套入门 SDK。

路线：定义材料 → 定义制品 → 纯策略 → 执行动作 → 新存储连接读回。

In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show

from powercontext.http import CreateScopeRequest

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))
if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("A1", features=())
client = lab.client
assert client is not None
scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · A1", summary="本次教学实验的独立材料", idempotency_key=f"{lab.run_id}:main"
    )
)
scope_id = scope.scope_id

## 自定义 Source：明确输入与读取结果

适配器只负责把应用输入变成 Source，并读取它的内容。它不负责生成报告或决定什么时候执行。我们捕获的是下面实际运行的 Decimal 检查。

In [ ]:
from decimal import Decimal

from pydantic import BaseModel

from powercontext import Source, SourceMaterialization, SourceRef


class CheckInput(BaseModel):
    run_id: str
    passed: int
    total: int


class CheckSource(Source):
    passed: int
    total: int


class CheckAdapter:
    name = "tutorial-check"
    input_class = CheckInput
    source_class = CheckSource

    async def resolve(self, value, /):
        return CheckSource(
            name=value.run_id, materialization=SourceMaterialization.CAPTURED, passed=value.passed, total=value.total
        )

    async def read(self, source, /):
        return {"passed": source.passed, "total": source.total}


checks = [Decimal("12.34") * 100 == 1234, Decimal("1.999") * 100 != 199]
adapter = CheckAdapter()
source = await adapter.resolve(CheckInput(run_id="actual-decimal-check", passed=sum(checks), total=len(checks)))
assert await adapter.read(source) == {"passed": 2, "total": 2}
show(source)

## 自定义 Artifact：让报告有准确版本与来源

Artifact 和 Draft 共享 family 与内容结构。报告保存检查数量；Sources 引用说明它根据哪份材料形成。名称注册本身不等于具备 Memory 的召回行为。

In [ ]:
from typing import ClassVar

from powercontext import Artifact, ArtifactDraft


class CheckContent(BaseModel):
    passed: int
    total: int
    summary: str


class CheckReport(Artifact[CheckContent]):
    family: ClassVar[str] = "tutorial-check-report"


class CheckReportDraft(ArtifactDraft[CheckContent]):
    family: ClassVar[str] = "tutorial-check-report"


source_ref = SourceRef(source_type=adapter.name, source_id=source.name)
draft = CheckReportDraft(
    content=CheckContent(passed=source.passed, total=source.total, summary="两个实际 Decimal 检查通过"),
    sources=(source_ref,),
)
show(draft)

## Trigger 只产生状态与动作

同一个检查信号不应重复触发报告。这个 Trigger 根据已见过的 run_id 产生纯 PolicyTransition；调用它不会写数据库。执行动作和持久化策略状态是组装方的责任。

In [ ]:
from powercontext import PolicyTransition, Trigger


class CheckSignal(BaseModel):
    run_id: str
    report: CheckReportDraft


class CheckTrigger:
    def initial_state(self):
        return ()

    def activate(self, signal, state, /):
        if signal.run_id in state:
            return PolicyTransition[tuple[str, ...], CheckReportDraft](state=state)
        return PolicyTransition[tuple[str, ...], CheckReportDraft](
            state=(*state, signal.run_id), actions=(signal.report,)
        )


trigger: Trigger[CheckSignal, tuple[str, ...], CheckReportDraft] = CheckTrigger()
signal = CheckSignal(run_id=source.name, report=draft)
transition = trigger.activate(signal, trigger.initial_state())
assert len(transition.actions) == 1
assert trigger.activate(signal, transition.state).actions == ()
show({"首次计划动作": len(transition.actions), "重复信号动作": 0})

## 组装方执行动作，真实写入数据库

关闭本篇 Server 后使用同一个实验数据库。我们注册 Source Adapter 和 Artifact 类型，在事务里先存来源，再存报告。这里显式选择内置关系存储扩展点，业务调用不能假定任何未注册类型都能被 Server 识别。

正式后台系统还应把策略状态与报告一起持久化；本篇只验证一次纯转换及其动作执行，不能把内存中的去重状态当作跨重启调度保证。

In [ ]:
from powercontext.builtin.persistence.artifacts import ArtifactRepository
from powercontext.builtin.persistence.oceanbase import OceanBaseConfig, OceanBaseProfile
from powercontext.builtin.persistence.sources import SourceRepository
from powercontext.builtin.persistence.sqlite import SQLiteProfile
from powercontext.builtin.persistence.tables import SHARED_TABLES

await lab.close()
profile_type = OceanBaseProfile if isinstance(lab.database, OceanBaseConfig) else SQLiteProfile
sources = SourceRepository((adapter,))
artifacts = ArtifactRepository((CheckReport,))
async with (
    profile_type.open(lab.database, tables=SHARED_TABLES) as profile,
    profile.database.transaction() as connection,
):
    stored = await sources.add(connection, scope_id, source)
    report = await artifacts.create(connection, scope_id, "actual-report", transition.actions[0])
assert stored.ref == source_ref and report.revision == 1
assert report.lineage.sources == (source_ref,)
show(report)

## 用新连接读回准确版本

重新打开数据库，不复用上面的 connection。报告内容与 Source 都应能准确恢复。到这里，扩展点已经通过实际持久化串起来，但没有假装提供尚未组装的 HTTP 或召回能力。

In [ ]:
async with (
    profile_type.open(lab.database, tables=SHARED_TABLES) as profile,
    profile.database.transaction() as connection,
):
    restored = await artifacts.get(connection, scope_id, report.as_ref())
    restored_source = await sources.get(connection, scope_id, source_ref)
assert restored == report
assert restored_source.value == source
show({"自定义 family": report.family, "Revision": report.revision, "新连接恢复": True})

## 练习与验收

给 CheckContent 增加 failed 字段，调整适配器、策略输入和报告内容，再运行。不要把纯 Trigger 的 actions 当成已执行的动作；实际存储结果才证明报告形成。

最后关闭服务。实验文件保留在本次 `.powercontext/` 目录，便于复查。

In [ ]:
await lab.close()
print("本篇 Server 已关闭。")